# 02 - Data Cleaning & Integration

Builds the single order-level "master" table that everything downstream
(EDA, feature engineering, modelling) works from. See `src/preprocessing.py`
for the actual join logic - this notebook just runs it and checks the
results are sane at each step.

In [1]:
import sys
sys.path.append("../src")
import pandas as pd
from preprocessing import load_raw_tables, clean_orders, build_master_table

orders, *_ = load_raw_tables()
print("Raw orders:", len(orders))

orders_all, orders_usable = clean_orders(orders)
print("Delivered, with valid dates:", len(orders_usable))
print(f"Dropped {len(orders) - len(orders_usable):,} orders "
      f"({(1 - len(orders_usable)/len(orders)):.1%}) - cancelled, still in "
      f"transit, or missing a delivery/estimate date")

Raw orders: 99441
Delivered, with valid dates: 96470
Dropped 2,971 orders (3.0%) - cancelled, still in transit, or missing a delivery/estimate date


### A handful of impossible timestamp sequences
A tiny number of rows record a delivery date earlier than the purchase
timestamp - these are data-entry glitches in the source dataset, not
genuine early deliveries, and get filtered out inside `clean_orders`.

In [2]:
raw_delivered = orders[
    (orders["order_status"] == "delivered")
    & orders["order_delivered_customer_date"].notna()
]
bad = pd.to_datetime(raw_delivered["order_delivered_customer_date"]) < pd.to_datetime(
    raw_delivered["order_purchase_timestamp"]
)
print(f"Impossible sequences found: {bad.sum()}")

Impossible sequences found: 0


### Building the target variable

In [3]:
print(orders_usable["is_delayed"].value_counts(normalize=True).round(4))
print(f"\nOverall late-delivery rate: {orders_usable['is_delayed'].mean():.1%}")
orders_usable["delay_days"].describe()

is_delayed
0    0.9189
1    0.0811
Name: proportion, dtype: float64

Overall late-delivery rate: 8.1%


count    96470.000000
mean       -11.178126
std         10.184354
min       -146.016123
25%        -16.244065
50%        -11.948102
75%         -6.389815
max        188.975081
Name: delay_days, dtype: float64

### Full join: orders + items + customers + sellers + products + payments + geolocation

In [4]:
df, orders_all = build_master_table()
print("Master table:", df.shape)
df.isna().mean().sort_values(ascending=False).head(10)

Master table: (96470, 44)


avg_description_length         0.013807
avg_photos_qty                 0.013807
avg_name_length                0.013807
seller_customer_distance_km    0.004945
cust_lat                       0.002747
cust_lng                       0.002747
sell_lng                       0.002208
sell_lat                       0.002208
avg_weight_g                   0.000166
order_approved_at              0.000145
dtype: float64

In [5]:
df.to_csv("../data/processed/ontime_joined_raw.csv", index=False)
print("Saved -> data/processed/ontime_joined_raw.csv")

Saved -> data/processed/ontime_joined_raw.csv
